# QAOA · the 144-point parameter landscape

A standalone recreation of the QAOA landscape: **12 β values × 12 γ values = 144 actual quantum-state evaluations**. All visualization code is contained in this notebook; it needs no companion helpers and does not modify `qlang.ipynb`.

Use a Python kernel compatible with qlang (Python 3.10+). Run **Setup**, the collapsed **Implementation** cell, then **Launch and run** (or Run All). The last cell contains a saved HTML/SVG preview, so the initial landscape is visible even before the widget renderer connects.


In [1]:
# Install into the notebook's active kernel. Git must be available.
%pip install "qlang @ git+https://github.com/asherhe/qlang.git" "ipywidgets>=8.1"


  Cloning https://github.com/asherhe/qlang.git to C:\Users\franc\AppData\Local\Temp\pip-install-x3r09zbc\qlang_2ecd5d8a5020429faeb386ab54221ab9
  Resolved https://github.com/asherhe/qlang.git to commit d7b79f5ce477b3cf64e4cdfc502d3a0a8edfc8d2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for qlang: filename=qlang-0.1.0-py3-none-any.whl size=38656 sha256=fb6d117a186d8830adf5caffe59aa599727806509bca4c9defcef60e4e1ec0f3
  Stored in directory: C:\Users\franc\AppData\Local\Temp\pip-ephem-wheel-cache-639dcbxb\wheels\f5\1c\e2\edd456de60bc591f9d368c8b4508e3b3993b06519114b622d9
Successfully built qlang
  Attempting uninstall: qlang
    Found existing installation: qlang 0.1.0
    Uninstalling qlang-0

  Running command git clone --filter=blob:none --quiet https://github.com/asherhe/qlang.git 'C:\Users\franc\AppData\Local\Temp\pip-install-x3r09zbc\qlang_2ecd5d8a5020429faeb386ab54221ab9'

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Design and interpretation

- **Horizontal axis:** β, the mixer angle. **Vertical axis:** γ, the cost angle. Both span `[0, 2π)`.
- **Color:** exact expected objective-plus-penalty cost. Dark blue is lower; the background is `#f8f7f2`.
- **Green outline:** the best angle pair on the tested grid. **Black outline:** the currently selected pair.
- **β/γ sliders:** inspect cached states without rerunning optimization. The readout shows expected combined cost, feasible probability, and mean objective conditional on feasibility.
- **Controls:** editable variables, constraint, unsigned objective, penalty, and an 8/12/16-point grid. Empty objective means feasibility only. Changes mark old results as stale; rerun explicitly. Unused declared variables do not count toward the six-active-bit limit.

The constraint and objective are parsed together using the exact same variable objects. Cost construction reuses qlang's encoding helpers and the same direct/fallback policy as `qlang.qaoa`; quantum states use its actual one-layer QAOA ansatz. The solver's β-first grid order is retained for tie-breaking. The full result (angles, expected costs, probability distributions, assignments, and feasibility) is shared by the plot and readouts.

For a supported linear equality, the penalty is its squared residual. Otherwise qlang uses a 0/1 violation penalty. The panel identifies the active encoding. The default penalty of 10 is adjustable and is not automatically sufficient for every problem. These are noiseless statevector expectations, not sampled frequencies, and the best grid point is not a guarantee of a globally optimal solution. No additional domain restrictions are imposed.


## Implementation

The following cell contains the complete, readable Python implementation. It is collapsed to keep the presentation compact. Expand it to inspect the shared parser, result model, cost adapter, SVG renderer, and dashboard lifecycle.


In [2]:
import ast
import asyncio
import html
import importlib
import keyword
from dataclasses import dataclass
from math import isfinite, pi

import ipywidgets as widgets
from IPython.display import HTML, display
import qlang


def parse_problem(declarations, constraint_text, objective_text):
    """Parse both expressions with one shared variable namespace; no eval."""
    names = [row[0] for row in declarations]
    if len(names) != len(set(names)):
        raise ValueError('Variable names must be unique.')
    if any(not name.isidentifier() or keyword.iskeyword(name) for name in names):
        raise ValueError('Use valid Python identifiers for variable names.')
    if any(kind not in ('UInt','Bit') or not 1 <= width <= 16 for _,kind,width in declarations):
        raise ValueError('Choose UInt or Boolean variables with widths from 1 to 16.')
    qlang.ASTNode.reset()
    variables = {name: qlang.uint(name,width) if kind=='UInt' else qlang.var_bit(name)
                 for name,kind,width in declarations}
    used = set()
    binary = {ast.Add:lambda a,b:a+b,ast.Sub:lambda a,b:a-b,ast.Mult:lambda a,b:a*b,
              ast.BitAnd:lambda a,b:a&b,ast.BitOr:lambda a,b:a|b,ast.BitXor:lambda a,b:a^b,
              ast.LShift:lambda a,b:a<<b,ast.RShift:lambda a,b:a>>b}
    compare = {ast.Eq:lambda a,b:a==b,ast.NotEq:lambda a,b:a!=b,ast.Lt:lambda a,b:a<b,
               ast.LtE:lambda a,b:a<=b,ast.Gt:lambda a,b:a>b,ast.GtE:lambda a,b:a>=b}
    def parse(text):
        if len(text)>500: raise ValueError('Keep expressions below 500 characters.')
        tree=ast.parse(text,mode='eval')
        if sum(1 for _ in ast.walk(tree))>150: raise ValueError('Expression is too large for this demo.')
        def build(node):
            if isinstance(node,ast.Name):
                if node.id not in variables: raise ValueError(f'Add variable {node.id!r} to the table.')
                used.add(node.id)
                return variables[node.id]
            if isinstance(node,ast.Constant) and type(node.value) in (int,bool) and 0<=node.value<=65535:
                return node.value
            if isinstance(node,ast.BinOp) and type(node.op) in binary:
                a,b=build(node.left),build(node.right)
                if isinstance(node.op,(ast.LShift,ast.RShift)) and (type(b) is not int or not 0<=b<=8):
                    raise ValueError('Use constant shifts from 0 to 8.')
                return binary[type(node.op)](a,b)
            if isinstance(node,ast.UnaryOp) and isinstance(node.op,ast.Invert):
                value=build(node.operand)
                if not isinstance(value,qlang.Bit): raise ValueError('Use ~ on Boolean expressions only.')
                return ~value
            if isinstance(node,ast.Compare) and len(node.ops)==1 and type(node.ops[0]) in compare:
                return compare[type(node.ops[0])](build(node.left),build(node.comparators[0]))
            raise ValueError('Use arithmetic, bitwise logic and parenthesized comparisons; calls, attributes and Python and/or/not are not supported.')
        result=build(tree.body)
        if type(result) is bool: return qlang.const_bit(result)
        if type(result) is int: return qlang.uint(result)
        return result
    constraint=parse(constraint_text)
    objective=parse(objective_text) if objective_text.strip() else None
    if not isinstance(constraint,qlang.Bit): raise ValueError('The constraint must be Boolean, such as x + y == 3.')
    if objective is not None and not isinstance(objective,qlang.UInt):
        raise ValueError('The objective must be an unsigned-integer cost; leave it blank for feasibility only.')
    widths={name:(width if kind=='UInt' else 1) for name,kind,width in declarations if name in used}
    if not 0<sum(widths.values())<=6: raise ValueError('Use 1–6 active input bits; unused declarations do not count.')
    return constraint,objective,widths


def landscape_cost_operator(constraint,widths,objective,penalty):
    """Reuse qlang's encoding helpers and direct/fallback selection policy."""
    core=importlib.import_module('qlang.qalgorithm.qaoa')
    if hasattr(core,'build_cost_operator'):
        return core.build_cost_operator(constraint,widths,objective,penalty)
    # Compatibility with qlang releases before the reusable wrapper was added.
    linear=core.evaluate_linear(constraint)
    if linear is NotImplemented:
        return core.constraint_to_cost_op(constraint,widths,objective,penalty)
    constant,coefficients=linear
    terms=core.squared_residuals(float(constant),core.expand_to_bits(coefficients,widths),penalty)
    if objective is not None:
        extracted=core.extract_linear(objective)
        if extracted is NotImplemented: raise ValueError('A linear objective is required for the direct linear-equality path.')
        constant,coefficients=extracted
        additions=core.linear_objective_terms(constant,core.expand_to_bits(coefficients,widths))
        for mask,coefficient in additions.items(): terms[mask]=terms.get(mask,0)+coefficient
    return core.terms_to_op(terms,sum(widths.values()))


@dataclass
class LandscapeResult:
    key: tuple
    widths: dict
    angles: tuple
    costs: tuple
    probabilities: tuple
    assignments: tuple
    feasible: tuple
    objectives: tuple
    best: tuple
    encoding: str


def compute_landscape(declarations,constraint_text,objective_text,penalty=10.0,resolution=12):
    if not isfinite(penalty) or penalty<=0: raise ValueError('Penalty must be finite and positive.')
    if resolution not in (8,12,16): raise ValueError('Choose an 8, 12, or 16 point grid.')
    constraint,objective,widths=parse_problem(declarations,constraint_text,objective_text)
    core=importlib.import_module('qlang.qalgorithm.qaoa')
    operator=landscape_cost_operator(constraint,widths,objective,penalty)
    state_at,_=core.qaoa_creator(operator)
    angles=tuple(i*2*pi/resolution for i in range(resolution))
    assignments=tuple(core.decode_state(i,widths) for i in range(1<<sum(widths.values())))
    feasible=tuple(bool(qlang.evaluate(constraint,row)) for row in assignments)
    objectives=tuple(0 if objective is None else qlang.evaluate(objective,row) for row in assignments)
    costs,probabilities=[],[]
    for gamma in angles:
        cost_row,probability_row=[],[]
        for beta in angles:
            state=state_at(beta,gamma)
            cost_row.append(float(state.expectation_value(operator).real))
            probability_row.append(tuple(float(p) for p in state.probabilities()))
        costs.append(tuple(cost_row));probabilities.append(tuple(probability_row))
    best=min(((b,g) for b in range(resolution) for g in range(resolution)),key=lambda pair:costs[pair[1]][pair[0]])
    encoding='squared residual (linear equality)' if core.evaluate_linear(constraint) is not NotImplemented else '0/1 constraint violation'
    key=(tuple(declarations),constraint_text,objective_text,penalty,resolution)
    return LandscapeResult(key,widths,angles,tuple(costs),tuple(probabilities),assignments,feasible,objectives,best,encoding)


def render_landscape(result,beta_index,gamma_index):
    size,left,top=36,72,48
    n=len(result.angles);low=min(map(min,result.costs));high=max(map(max,result.costs))
    width,height=left+n*size+120,top+n*size+60
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img" aria-label="QAOA parameter cost landscape">',
           '<rect width="100%" height="100%" fill="#f8f7f2"/>',
           f'<text x="{left}" y="20" font-family="system-ui" font-size="14">Expected combined cost · darker is lower</text>']
    for g,row in enumerate(result.costs):
        for b,cost in enumerate(row):
            fraction=(cost-low)/(high-low) if high>low else 0
            rgb=tuple(round(a+(z-a)*fraction) for a,z in zip((29,78,216),(248,247,242)))
            fill='#%02x%02x%02x'%rgb
            tip=html.escape(f'β={result.angles[b]:.4f}, γ={result.angles[g]:.4f}; expected cost={cost:.6f}')
            parts.append(f'<rect x="{left+b*size}" y="{top+g*size}" width="{size}" height="{size}" fill="{fill}" stroke="#f8f7f2" stroke-width="1"><title>{tip}</title></rect>')
    for i,angle in enumerate(result.angles):
        if i%max(1,n//4)==0:
            parts.append(f'<text x="{left+(i+.5)*size}" y="{top+n*size+18}" text-anchor="middle" font-family="system-ui" font-size="10">{angle:.2f}</text>')
            parts.append(f'<text x="{left-10}" y="{top+(i+.6)*size}" text-anchor="end" font-family="system-ui" font-size="10">{angle:.2f}</text>')
    for b,g,color,inset in [(*result.best,'#16a34a',3),(beta_index,gamma_index,'#111827',7)]:
        for stroke,thickness in [('white',5),(color,2)]:
            parts.append(f'<rect x="{left+b*size+inset}" y="{top+g*size+inset}" width="{size-2*inset}" height="{size-2*inset}" fill="none" stroke="{stroke}" stroke-width="{thickness}"/>')
    parts.extend([f'<text x="{left+n*size/2}" y="{height-10}" text-anchor="middle" font-family="system-ui" font-size="13">β · mixer angle (radians)</text>',
                  f'<text x="15" y="{top+n*size/2}" transform="rotate(-90 15 {top+n*size/2})" text-anchor="middle" font-family="system-ui" font-size="13">γ · cost angle (radians)</text>',
                  f'<text x="{left+n*size+10}" y="{top+15}" font-family="system-ui" font-size="11">Min: {low:.3f}</text>',
                  f'<text x="{left+n*size+10}" y="{top+34}" font-family="system-ui" font-size="11">Max: {high:.3f}</text>','</svg>'])
    return '<div style="overflow:auto">'+''.join(parts)+'</div>'


def result_details(result,b,g):
    probabilities=result.probabilities[g][b]
    feasible_probability=sum(p for p,good in zip(probabilities,result.feasible) if good)
    weighted=sum(p*value for p,value,good in zip(probabilities,result.objectives,result.feasible) if good)
    conditional=f'{weighted/feasible_probability:.4f}' if feasible_probability>1e-12 else 'undefined (no feasible probability)'
    return (f'<p><b>Selected β={result.angles[b]:.4f}, γ={result.angles[g]:.4f}</b><br>'
            f'Expected combined cost: {result.costs[g][b]:.4f} · Feasible probability: {feasible_probability:.1%}<br>'
            f'Mean objective conditional on feasibility: {conditional}</p>')


class QAOALandscapeDashboard:
    def __init__(self):
        self.rows=[];self.result=None;self.task=None;self.closed=False;self.updating=True
        self.constraint=widgets.Text(value='(x + y == 7) & (x > y)',description='Constraint',continuous_update=True,layout=widgets.Layout(width='100%'))
        self.objective=widgets.Text(value='2 * x + y',description='Objective',continuous_update=True,layout=widgets.Layout(width='100%'))
        self.penalty=widgets.BoundedFloatText(value=10,min=.001,max=100000,description='Penalty',layout=widgets.Layout(width='190px'))
        self.resolution=widgets.Dropdown(options=[8,12,16],value=12,description='Grid',layout=widgets.Layout(width='150px'))
        self.run=widgets.Button(description='Run QAOA landscape',button_style='primary',layout=widgets.Layout(width='190px'))
        self.best_button=widgets.Button(description='Select best tested',disabled=True)
        self.beta=widgets.IntSlider(value=0,min=0,max=11,description='β index',continuous_update=False)
        self.gamma=widgets.IntSlider(value=0,min=0,max=11,description='γ index',continuous_update=False)
        self.table=widgets.VBox(layout=widgets.Layout(gap='6px'))
        self.note,self.plot,self.details=widgets.HTML(),widgets.HTML(),widgets.HTML()
        for name in ('x','y'):self.add_variable(name)
        add=widgets.Button(description='+ Add variable')
        add.on_click(self.add_variable)
        headers=widgets.HBox([widgets.HTML('<b>Name</b>',layout=widgets.Layout(width='110px')),widgets.HTML('<b>Type</b>',layout=widgets.Layout(width='95px')),widgets.HTML('<b>Bits</b>')])
        self.view=widgets.VBox([
            widgets.HTML('<style>.qaoa-standalone,.qaoa-standalone div,.qaoa-standalone input,.qaoa-standalone select,.qaoa-standalone button{background-color:#f8f7f2}.qaoa-stale{opacity:.35}</style><h2>QAOA parameter landscape</h2>'),
            self.constraint,self.objective,widgets.HTML('<b>Variables</b>'),headers,self.table,add,
            widgets.HBox([self.penalty,self.resolution,self.run],layout=widgets.Layout(flex_flow='row wrap')),
            self.note,self.plot,widgets.HBox([self.beta,self.gamma,self.best_button],layout=widgets.Layout(flex_flow='row wrap')),self.details,
            widgets.HTML('<small>Green: lowest expected cost on this grid. Black: selected angles. Sliders use cached states. Exact simulator expectations; no measurement shots, noise, or guarantee of a global optimum. One cost-and-mixer layer; six active input bits maximum. Blank objective means feasibility only. No extra domain restrictions are applied.</small>'),
        ],layout=widgets.Layout(width='100%',padding='12px',border='1px solid #cbd5e1'))
        self.view.add_class('qaoa-standalone')
        for control in (self.constraint,self.objective,self.penalty,self.resolution):control.observe(self.invalidate,names='value')
        for control in (self.beta,self.gamma):control.observe(self.redraw,names='value')
        self.run.on_click(self.request_run);self.best_button.on_click(self.select_best)
        self.updating=False;self.invalidate()

    def add_variable(self,name=None):
        if not isinstance(name,str):
            used={row[0].value for row in self.rows};i=1
            while f'v{i}' in used:i+=1
            name=f'v{i}'
        if len(self.rows)>=16:return
        field=widgets.Text(value=name,continuous_update=True,layout=widgets.Layout(width='110px'))
        kind=widgets.Dropdown(options=[('UInt','UInt'),('Boolean','Bit')],layout=widgets.Layout(width='95px'))
        bits=widgets.BoundedIntText(value=3,min=1,max=16,layout=widgets.Layout(width='65px'))
        remove=widgets.Button(description='×',layout=widgets.Layout(width='34px'))
        row=(field,kind,bits,remove,widgets.HBox([field,kind,bits,remove]))
        self.rows.append(row);self.table.children=tuple(r[4] for r in self.rows)
        def set_type(change):
            bits.disabled=change['new']=='Bit'
            if bits.disabled:bits.value=1
            self.invalidate()
        def remove_row(_):
            self.rows.remove(row);self.table.children=tuple(r[4] for r in self.rows)
            for control in row:control.close()
            self.invalidate()
        field.observe(self.invalidate,names='value');kind.observe(set_type,names='value');bits.observe(self.invalidate,names='value');remove.on_click(remove_row)
        self.invalidate()

    def key(self):
        return tuple((r[0].value,r[1].value,r[2].value) for r in self.rows),self.constraint.value,self.objective.value,self.penalty.value,self.resolution.value

    def invalidate(self,change=None):
        if self.updating or self.closed:return
        self.note.value='<p>Settings changed. Click Run QAOA landscape. Any previous plot is stale.</p>'
        self.plot.add_class('qaoa-stale');self.best_button.disabled=True

    def accept(self,result):
        if self.closed or result.key!=self.key():return
        self.result=result;self.updating=True
        self.beta.max=self.gamma.max=len(result.angles)-1
        self.beta.value,self.gamma.value=result.best
        self.updating=False;self.plot.remove_class('qaoa-stale');self.best_button.disabled=False;self.redraw()

    def run_sync(self):
        result=compute_landscape(*self.key());self.accept(result);return result

    def request_run(self,_=None):
        if self.closed:return
        if self.result is not None and self.result.key==self.key():self.select_best();return
        try:loop=asyncio.get_running_loop()
        except RuntimeError:
            try:self.run_sync()
            except Exception as error:self.note.value='<p style="color:#b45309">'+html.escape(str(error))+'</p>'
        else:
            if self.task is None or self.task.done():self.task=loop.create_task(self.run_async(self.key()))

    async def run_async(self,key):
        self.run.disabled=True;self.note.value=f'<p>Evaluating {key[-1]**2} angle pairs…</p>'
        try:
            result=await asyncio.to_thread(compute_landscape,*key)
            self.accept(result)
            if not self.closed and result.key!=self.key():self.invalidate()
        except Exception as error:
            if not self.closed:self.note.value='<p style="color:#b45309">'+html.escape(str(error))+'</p>'
        finally:
            if not self.closed:self.run.disabled=False

    def select_best(self,_=None):
        if self.result is None or self.result.key!=self.key():return
        self.updating=True;self.beta.value,self.gamma.value=self.result.best;self.updating=False;self.redraw()

    def redraw(self,change=None):
        if self.updating or self.closed or self.result is None:return
        if self.result.key!=self.key():self.invalidate();return
        r,b,g=self.result,self.beta.value,self.gamma.value
        self.plot.value=render_landscape(r,b,g)
        self.note.value=f'<p><b>{len(r.angles)**2} angle pairs evaluated</b> · {sum(r.widths.values())} input qubits · Lowest tested expected cost: {r.costs[r.best[1]][r.best[0]]:.4f}<br>Penalty encoding: {r.encoding}</p>'
        self.details.value=result_details(r,b,g)

    def snapshot_html(self):
        if self.result is None:return '<p>Run the dashboard to create a preview.</p>'
        r=self.result;b,g=r.best
        return '<div style="background:#f8f7f2;padding:12px"><h3>Saved landscape preview</h3>'+self.note.value+render_landscape(r,b,g)+result_details(r,b,g)+'</div>'

    def close(self):
        self.closed=True
        if self.task is not None:self.task.cancel()
        self.view.close()


## Launch and run

The first run evaluates the original example, `(x + y == 7) & (x > y)`, with objective `2 * x + y`, penalty 10, and a 12×12 grid. Subsequent edits run only when you click **Run QAOA landscape**. **Select best tested** returns the selectors to the best tested angles.

Rerunning this cell closes the previous dashboard before creating another. If widget controls are blank after installing packages, restart the kernel and run again. JupyterLab provides a bundled widget renderer; VS Code requires its Jupyter widget support.


In [3]:
if "landscape" in globals():
    landscape.close()
landscape = QAOALandscapeDashboard()
display(landscape.view)
initial_result = landscape.run_sync()  # Actually evaluate all 144 angle pairs.
print(f"Completed {len(initial_result.angles)**2} QAOA angle pairs.")


Completed 144 QAOA angle pairs.


## Saved preview

This static snapshot uses the same result and SVG renderer as the live controls. It is saved with the notebook for easy viewing and does not respond to widget edits. Rerun the cell to refresh it from the current result.


In [4]:
display(HTML(landscape.snapshot_html()))
